# TMT data treatment

This code will work with the TMT data provided by the FGCZ.

Initial working file for each experiment (hTERT_HME1_1, hTERT_HME1_2, HEK293T_1) is the raw_abundances_matrix

**First step is to rename the files to have a unified labeling system** -> (CellLine)_ (Data:Type)_ (Treatment)_ (TimePoint)_ (Replicate)

All the data transformation and statistics I am going to do starting from that initial "raw_data" file.



In [1]:
from src.transformations import *  # all necessary pakages are imported within src.transformations

# hme1_1 = pd.read_csv("../../data/hme1_1_raw_sample.tsv", sep="\t")
# hme1_2 = pd.read_csv("../../data/hme1_2_raw_sample.tsv", sep="\t")
# hek_1 = pd.read_csv("../../data/hek_1_raw_sample.tsv", sep="\t")
#
hme1_1 = pd.read_csv("../../Experiment/hme1_1/Data/All/20260427_hTERT_HME1_1.tsv", sep="\t")
hme1_2 = pd.read_csv("../../Experiment/hme1_2/Data/All/20260427_hTERT_HEM1_2.tsv", sep="\t")
hek_1 = pd.read_csv("../../Experiment/hek_1/Data/All/20260417_HEK293T_raw_selection.tsv", sep="\t")

## Data transformations

Run the full transformation pipeline on each dataset:
- n:reps  # important to double check this number of reps identified for the LFQ dataset
- raw:mean / raw:median / raw:sd / raw:cv
- log2:abs (zeros treated as NaN)
- log2:mean / log2:median / log2:sd
- log2:FC (fold change vs. starve)
- log2:scaled (max-normalised fold change, amplitude between -1 and 1)
- log2:pvalue (Welch t-test vs. starve)
- log2:FDR (Benjamini-Hochberg correction per treatment × timepoint)
- log2:adjustedFDR (-log10 transformation of the FDR, also called adjusted FDR or adjusted p-value)


In [2]:
hme1_1_transformed = run_all_transformations(hme1_1, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hme1_2_transformed = run_all_transformations(hme1_2, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hek_1_transformed  = run_all_transformations(hek_1,  cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])

print(f"hme1_1: {hme1_1.shape} -> {hme1_1_transformed.shape}")
print(f"hme1_2: {hme1_2.shape} -> {hme1_2_transformed.shape}")
print(f"hek_1:  {hek_1.shape}  -> {hek_1_transformed.shape}")

[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 328 columns.

All cell lines processed. Total columns: 90 -> 418
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 328 columns.

All cell lines processed. Total columns: 106 -> 434
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 328 columns.

All cell lines processed. Total columns: 106 -> 434
hme1_1: (35002, 90) -> (35002, 418)
hme1_2: (50002, 106) -> (50002, 434)
hek_1:  (36567, 106)  -> (36567, 434)


In [3]:
# Save transformed datasets
# hek_1_transformed.to_csv("../../Experiment/hek_1/Data/Processed/20260709_HEK293T_processed.tsv", sep="\t", index=False)
# hme1_1_transformed.to_csv("../../Experiment/hme1_1/Data/Processed/20260709_hTERT_HEM1_1_processed.tsv", sep="\t", index=False)
# hme1_2_transformed.to_csv("../../Experiment/hme1_2/Data/Processed/20260709_hTERT_HEM1_2_processed.tsv", sep="\t", index=False)

## Merging external data

PhosphoSite Plus dataset

In [4]:
functional_score_df = pd.read_csv("../../External_Data/Metadata/PhosphoSitePlus.tsv", sep="\t")
regulatory_sites = pd.read_csv("../../External_Data/Metadata/Phosphosite/Regulatory_sites.tsv", sep="\t")

print(f"functional_score_df columns: {functional_score_df.columns}")
print(f"regulatory_sites columns: {regulatory_sites.columns}")

functional_score_df columns: Index(['protein_Id', 'protein_name', 'prot_seq_position', 'aa', 'site',
       'functional_score', 'ms_lit', 'ERK_motif', 'ERK_ext_motif'],
      dtype='object')
regulatory_sites columns: Index(['GENE', 'protein_name', 'information', 'protein_Id', 'GENE_ID',
       'HU_CHR_LOC', 'ORGANISM', 'MOD_RSD', 'SITE_GRP_ID', 'SITE_+/-7_AA',
       'DOMAIN', 'ON_FUNCTION', 'ON_PROCESS', 'ON_PROT_INTERACT',
       'ON_OTHER_INTERACT', 'PMIDs', 'LT_LIT', 'MS_LIT', 'MS_CST', 'NOTES'],
      dtype='object')


In [9]:
df_with_extra_info = merge_functional_score(df = hme1_2_transformed,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position")
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ERK_motif"])
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = regulatory_sites,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ORGANISM", "ON_FUNCTION", "ON_PROCESS", "ON_PROT_INTERACT", 'ON_OTHER_INTERACT'],
                                            regulatory_sites= True)
df_with_extra_info

,protein_Id,nrPeptides,description,protein_name,protein_length,nr_tryptic_peptides,peptide_index,peptide_seq,SequenceWindow,Start,...,WT_log2:adjustedFDR_EGFnINS_10,WT_log2:adjustedFDR_EGFnINS_15,WT_log2:adjustedFDR_EGFnINS_90,functional_score,ERK_motif,ORGANISM,ON_FUNCTION,ON_PROCESS,ON_PROT_INTERACT,ON_OTHER_INTERACT
0,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_0,SGSGNFGGGR,SSSQRGR.SGSGNFGGGR.GGGFGGN,197,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_1_S199,SGsGNFGGGR,SSSQRGR.SGsGNFGGGR.GGGFGGN,197,...,1.455728,1.032540,1.232910,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_2_6_1_1_S6,SKSEsPKEPEQLR,.SKSEsPKEPEQLR.KIFIGGI,2,...,0.390373,0.243264,0.439499,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_2_6_2_2_S2S4,sKsESPK,.sKsESPK.EPEQIRK,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_305_316_1_0,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,QYFAKPR.NQGGYGGSSSSSSYGSGR.RF;QYFAKPR.NQGGYGGS...,301,...,2.186120,0.554312,1.384141,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49997,Q9Y3T6,1,R3H and coiled-coil domain-containing protein ...,R3HCC1,440.0,20.0,Q9Y3T6_236_237_1_0,FGSTLQLDLEK,MVEMATR.FGSTLQLDLEK.GKESIIE,234,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49998,Q9Y546,1,Leucine-rich repeat-containing protein 42 OS=H...,LRRC42,428.0,23.0,Q9Y546_384_388_1_0,HEAISSQESKK,CHGPVIK.HEAISSQESKK.SKKRPFE,380,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49999,Q9Y592,1,Centrosomal protein of 83 kDa OS=Homo sapiens ...,CEP83,701.0,31.0,Q9Y592_698_699_1_1_S698,KQLEELGsSGE,IETTQRK.KQLEELGsSGE.,691,...,NaN,NaN,NaN,NaN,False,human,intracellular localization,cytoskeletal reorganization,NaN,NaN
50000,Q9Y5J5,1,Pleckstrin homology-like domain family A membe...,PHLDA3,127.0,9.0,Q9Y5J5_119_127_1_1_S119,QsLGTGTLVS,IQTVRAR.QsLGTGTLVS.,118,...,NaN,NaN,NaN,0.416132,False,NaN,NaN,NaN,NaN,NaN


In [11]:
# df_with_extra_info.to_csv("../../Experiment/hek_1/Data/Processed/20260709_HEK293T_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_1/Data/Processed/20260709_hTERT_HEM1_1_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_2/Data/Processed/20260709_hTERT_HEM1_2_processed_phPlus.tsv", sep="\t", index=False)

In [12]:
# df_with_extra_info.loc[df_with_extra_info["protein_name"] == "EGFR"]